<a href="https://colab.research.google.com/github/abhisheksvcs25-bms/Bio-Inspired-Systems/blob/YT-Hand-Gesture-Controller/Parallel_Cellular_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
from concurrent.futures import ThreadPoolExecutor

np.random.seed(42)

# Create a 5G cellular network
GRID_SIZE = 20
x = np.linspace(0, 1000, GRID_SIZE)
xx, yy = np.meshgrid(x, x)
grid = np.column_stack((xx.ravel(), yy.ravel()))

bs = np.array([
    [200, 200],
    [800, 200],
    [200, 800],
    [800, 800]
])

N_BS = len(bs)
NOISE_DBM = -104
POWER_MIN, POWER_MAX = 20, 46

# Initial parameters: transmission powers and antenna directions
initial_power = np.full(N_BS, 43.0)
initial_angles = np.array([45.0, 135.0, 315.0, 225.0])

# Parameters are normalized between 0 and 1
initial = np.concatenate([
    (initial_power - POWER_MIN) / (POWER_MAX - POWER_MIN),
    initial_angles / 360
])

def calculate_coverage(parameters, cells):
    powers = POWER_MIN + parameters[:N_BS] * (POWER_MAX - POWER_MIN)
    angles = parameters[N_BS:] * 360

    distances = np.linalg.norm(
        cells[:, None, :] - bs[None, :, :], axis=2
    )
    distances = np.maximum(distances, 10)

    path_loss = (
        32.4 + 20 * np.log10(3500)
        + 20 * np.log10(distances / 1000)
    )

    directions = np.degrees(np.arctan2(
        cells[:, None, 1] - bs[None, :, 1],
        cells[:, None, 0] - bs[None, :, 0]
    ))

    difference = (directions - angles + 180) % 360 - 180
    antenna_gain = 8 * np.cos(np.radians(difference)) ** 2

    received = powers[None, :] - path_loss + antenna_gain

    received_linear = 10 ** (received / 10)
    noise_linear = 10 ** (NOISE_DBM / 10)

    total = np.sum(received_linear, axis=1, keepdims=True)
    sinr = received_linear / (
        total - received_linear + noise_linear
    )

    sinr_db = 10 * np.log10(np.maximum(sinr, 1e-15))

    best_signal = np.max(received, axis=1)
    best_sinr = np.max(sinr_db, axis=1)

    covered = (best_signal >= -100) & (best_sinr >= -5)
    return covered.astype(float)

# Parallel evaluation of network coverage
def objective(parameters):
    chunks = np.array_split(grid, 4)

    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(
            lambda cells: calculate_coverage(parameters, cells),
            chunks
        ))

    return np.mean(np.concatenate(results))

# Zeroth-order coordinate gradient estimation
def estimate_gradient(parameters, epsilon=0.01):
    gradient = np.zeros_like(parameters)

    for i in range(len(parameters)):
        plus = parameters.copy()
        minus = parameters.copy()

        plus[i] = min(1, plus[i] + epsilon)
        minus[i] = max(0, minus[i] - epsilon)

        gradient[i] = (
            objective(plus) - objective(minus)
        ) / (plus[i] - minus[i] + 1e-12)

    return gradient

# ZO-PGD algorithm
def zo_pgd(parameters, iterations=30, learning_rate=0.08):
    parameters = parameters.copy()

    for i in range(iterations):
        gradient = estimate_gradient(parameters)
        parameters = np.clip(
            parameters + learning_rate * gradient, 0, 1
        )
        print(f"ZO-PGD Iteration {i+1}: Coverage = {objective(parameters):.4f}")

    return parameters

# ZO-BCPGD algorithm
def zo_bcpgd(parameters, iterations=30, learning_rate=0.08):
    parameters = parameters.copy()
    blocks = [
        np.arange(N_BS),
        np.arange(N_BS, 2 * N_BS)
    ]

    for i in range(iterations):
        for block in blocks:
            gradient = estimate_gradient(parameters)
            parameters[block] += learning_rate * gradient[block]
            parameters = np.clip(parameters, 0, 1)

        print(f"ZO-BCPGD Iteration {i+1}: Coverage = {objective(parameters):.4f}")

    return parameters

# Run optimization
print("Initial Network Coverage:", round(objective(initial), 4))

print("\nRunning ZO-PGD...")
pgd_result = zo_pgd(initial)

print("\nRunning ZO-BCPGD...")
bcpgd_result = zo_bcpgd(initial)

print("\nFinal Results")
print("---------------------------")
print("Initial Coverage :", round(objective(initial), 4))
print("ZO-PGD Coverage  :", round(objective(pgd_result), 4))
print("ZO-BCPGD Coverage:", round(objective(bcpgd_result), 4))

print("\nOptimized ZO-PGD Parameters")
print("Transmission Powers (dBm):",
      np.round(POWER_MIN + pgd_result[:N_BS] * (POWER_MAX - POWER_MIN), 2))
print("Antenna Directions (degrees):",
      np.round(pgd_result[N_BS:] * 360, 2))

print("\nOptimized ZO-BCPGD Parameters")
print("Transmission Powers (dBm):",
      np.round(POWER_MIN + bcpgd_result[:N_BS] * (POWER_MAX - POWER_MIN), 2))
print("Antenna Directions (degrees):",
      np.round(bcpgd_result[N_BS:] * 360, 2))


Initial Network Coverage: 1.0

Running ZO-PGD...
ZO-PGD Iteration 1: Coverage = 1.0000
ZO-PGD Iteration 2: Coverage = 1.0000
ZO-PGD Iteration 3: Coverage = 1.0000
ZO-PGD Iteration 4: Coverage = 1.0000
ZO-PGD Iteration 5: Coverage = 1.0000
ZO-PGD Iteration 6: Coverage = 1.0000
ZO-PGD Iteration 7: Coverage = 1.0000
ZO-PGD Iteration 8: Coverage = 1.0000
ZO-PGD Iteration 9: Coverage = 1.0000
ZO-PGD Iteration 10: Coverage = 1.0000
ZO-PGD Iteration 11: Coverage = 1.0000
ZO-PGD Iteration 12: Coverage = 1.0000
ZO-PGD Iteration 13: Coverage = 1.0000
ZO-PGD Iteration 14: Coverage = 1.0000
ZO-PGD Iteration 15: Coverage = 1.0000
ZO-PGD Iteration 16: Coverage = 1.0000
ZO-PGD Iteration 17: Coverage = 1.0000
ZO-PGD Iteration 18: Coverage = 1.0000
ZO-PGD Iteration 19: Coverage = 1.0000
ZO-PGD Iteration 20: Coverage = 1.0000
ZO-PGD Iteration 21: Coverage = 1.0000
ZO-PGD Iteration 22: Coverage = 1.0000
ZO-PGD Iteration 23: Coverage = 1.0000
ZO-PGD Iteration 24: Coverage = 1.0000
ZO-PGD Iteration 25: Cov